# Semana 18 — Langfuse: dashboard visual y versionado de prompts

**Objetivo:** conectar ResearchAgent con Langfuse para tener trazas visuales,
comparar versiones de prompts y añadir puntuaciones de calidad.

**Al finalizar tendrás:**
- ResearchAgent enviando trazas a Langfuse en cada ejecución
- El prompt de síntesis gestionado desde el panel de Langfuse
- Puntuaciones de calidad vinculadas a cada traza
- Un `LangfuseTracer` que convive con el `Tracer` propio de la semana 17

**Prerrequisito de plataforma (antes de ejecutar):**
1. Crea una cuenta gratuita en https://cloud.langfuse.com  
   (o arranca self-hosted: `docker compose up` con el docker-compose de Langfuse)
2. Ve a Settings → API Keys → Create new keys
3. Añade al `.env` de `proyectos/research-agent/`:
   ```
   LANGFUSE_PUBLIC_KEY=pk-lf-...
   LANGFUSE_SECRET_KEY=sk-lf-...
   LANGFUSE_HOST=https://cloud.langfuse.com
   ```

**Si no quieres crear cuenta ahora:** el notebook tiene un modo `LANGFUSE_ENABLED=False`
que ejecuta todo localmente usando solo el `Tracer` de la semana 17.

**Coste estimado:** $0.02–0.05 (2-3 ejecuciones del agente).

## Celda 0 — Setup y detección de credenciales

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "langfuse>=2.0.0", "python-dotenv"])

import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_DIR = Path("../proyectos/research-agent").resolve()
load_dotenv(PROJECT_DIR / ".env")

sys.path.insert(0, str(PROJECT_DIR))

assert os.getenv("OPENAI_API_KEY"), "Define OPENAI_API_KEY en el .env"

# Langfuse es opcional — el notebook funciona sin credenciales
LANGFUSE_PUBLIC = os.getenv("LANGFUSE_PUBLIC_KEY", "")
LANGFUSE_SECRET = os.getenv("LANGFUSE_SECRET_KEY", "")
LANGFUSE_HOST   = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
LANGFUSE_ENABLED = bool(LANGFUSE_PUBLIC and LANGFUSE_SECRET)

if LANGFUSE_ENABLED:
    print("✓ Credenciales Langfuse encontradas — modo conectado")
    print(f"  Host: {LANGFUSE_HOST}")
else:
    print("⚠  Sin credenciales Langfuse — modo local (Tracer propio)")
    print("   Para conectar: añade LANGFUSE_PUBLIC_KEY y LANGFUSE_SECRET_KEY al .env")

## Celda 1 — Verificar la conexión con Langfuse

In [ ]:
if LANGFUSE_ENABLED:
    from langfuse import Langfuse

    langfuse = Langfuse(
        public_key=LANGFUSE_PUBLIC,
        secret_key=LANGFUSE_SECRET,
        host=LANGFUSE_HOST,
    )

    ok = langfuse.auth_check()
    if ok:
        print("✓ Conexión con Langfuse verificada")
        print(f"  Panel: {LANGFUSE_HOST}")
    else:
        print("✗ Error de autenticación — verifica las credenciales")
        LANGFUSE_ENABLED = False
else:
    langfuse = None
    print("Saltando verificación (modo local)")

## Celda 2 — Traza manual: el modelo conceptual

Antes de integrar con el agente, aprende el modelo Trace → Span → Generation
enviando una traza simple a mano.

In [ ]:
import time

if LANGFUSE_ENABLED:
    # ── Crea una traza completa a mano ────────────────────────────────────────
    trace = langfuse.trace(
        name="demo_manual",
        input={"tema": "Prueba de conexión"},
        metadata={"entorno": "notebook", "semana": 18},
    )

    # Span para una operación genérica (búsqueda web)
    span_busqueda = trace.span(
        name="buscar_web",
        input={"query": "IA en pesca galicia"},
    )
    time.sleep(0.3)   # simula latencia
    span_busqueda.end(
        output={"n_resultados": 5},
        metadata={"fuente": "duckduckgo"},
    )

    # Generation para una llamada al LLM
    gen = trace.generation(
        name="planificar",
        model="gpt-4o-mini",
        input=[{"role": "user", "content": "Planifica la investigación sobre IA en pesca"}],
        output="1. Buscar aplicaciones en acuicultura\n2. Revisar proyectos europeos",
        usage={"input": 450, "output": 180},
    )
    gen.end()

    # Cierra la traza con el resultado final
    trace.update(
        output={"n_subtareas": 2},
        tags=["demo", "semana-18"],
    )

    # IMPORTANTE: flush envía los datos antes de que termine el proceso
    langfuse.flush()

    print("✓ Traza enviada a Langfuse")
    print(f"  Búscala en: {LANGFUSE_HOST}/traces")
    print(f"  ID de traza: {trace.id}")

else:
    # Modo local: muestra la estructura sin enviar
    print("[Modo local] Estructura de una traza Langfuse:")
    estructura = {
        "trace": {
            "name": "investigar",
            "input": {"tema": "..."},
            "spans": [
                {"name": "buscar_web", "tipo": "Span",
                 "input": {"query": "..."}, "output": {"n_resultados": 5}},
                {"name": "planificar", "tipo": "Generation",
                 "model": "gpt-4o-mini", "usage": {"input": 450, "output": 180}},
            ],
            "output": {"informe": "..."},
        }
    }
    import json
    print(json.dumps(estructura, indent=2, ensure_ascii=False))

## Celda 3 — `LangfuseTracer`: wrapper que envuelve al `Tracer` propio

En lugar de reescribir `agent.py`, creamos un wrapper que usa el `Tracer` propio como fuente
de verdad y envía los datos a Langfuse al finalizar. Si Langfuse no está disponible,
se degrada silenciosamente al comportamiento local.

In [ ]:
LF_TRACER_PY = '''
"""
LangfuseTracer: extiende el Tracer propio enviando datos a Langfuse.
Si Langfuse no está configurado, funciona como el Tracer local.
"""
import os
from observability.tracer import Tracer, Span


class LangfuseTracer(Tracer):
    """
    Tracer con integración opcional de Langfuse.
    Hereda todo el comportamiento del Tracer local y añade el envío a Langfuse.

    Uso:
        tracer = LangfuseTracer(tema="...")  # Langfuse se activa si hay credenciales
        # El resto del código es idéntico al Tracer local
    """

    def __init__(self, tema: str, trazas_dir: str = "./trazas", tags: list[str] = None):
        super().__init__(tema=tema, trazas_dir=trazas_dir)
        self._lf_trace = None
        self._lf       = None
        self._tags     = tags or []
        self._init_langfuse(tema)

    def _init_langfuse(self, tema: str):
        """Inicializa Langfuse si las credenciales están disponibles."""
        pk = os.getenv("LANGFUSE_PUBLIC_KEY")
        sk = os.getenv("LANGFUSE_SECRET_KEY")
        if not (pk and sk):
            return
        try:
            from langfuse import Langfuse
            self._lf = Langfuse(
                public_key=pk,
                secret_key=sk,
                host=os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com"),
            )
            self._lf_trace = self._lf.trace(
                id=self.trace_id,
                name="investigar",
                input={"tema": tema},
                tags=self._tags,
            )
        except Exception as e:
            # No interrumpe la ejecución si Langfuse no está disponible
            self._lf = None
            self._lf_trace = None

    def registrar_generation(
        self,
        nombre: str,
        modelo: str,
        input_msgs: list[dict],
        output_txt: str,
        tokens_input: int,
        tokens_output: int,
        latencia_ms: float,
        prompt_name: str | None = None,
    ):
        """
        Registra una llamada al LLM como Generation en Langfuse
        (y la añade también al Tracer local como Span).
        """
        import time as _time
        s = Span(nombre=nombre, modelo=modelo)
        s.inicio          = _time.perf_counter() - latencia_ms / 1000
        s.fin             = _time.perf_counter()
        s.tokens_input    = tokens_input
        s.tokens_output   = tokens_output
        s.ok              = True
        with self._lock:
            self.spans.append(s)

        if self._lf_trace:
            try:
                kwargs = dict(
                    name=nombre,
                    model=modelo,
                    input=input_msgs,
                    output=output_txt,
                    usage={"input": tokens_input, "output": tokens_output},
                    metadata={"latencia_ms": round(latencia_ms, 1)},
                )
                if prompt_name:
                    kwargs["prompt"] = self._lf.get_prompt(prompt_name)
                gen = self._lf_trace.generation(**kwargs)
                gen.end()
            except Exception:
                pass

    def puntuar(self, nombre: str, valor: float, comentario: str = ""):
        """Añade una puntuación de calidad a la traza en Langfuse."""
        if self._lf_trace:
            try:
                self._lf_trace.score(
                    name=nombre,
                    value=valor,
                    comment=comentario,
                )
            except Exception:
                pass

    def cerrar(self, informe: str = "", coste_usd: float = 0.0):
        """Cierra la traza en Langfuse y guarda la traza local."""
        if self._lf_trace:
            try:
                self._lf_trace.update(
                    output={
                        "informe_preview": informe[:500] if informe else "",
                        "coste_usd": round(coste_usd, 6),
                        "tokens_totales": self.tokens_totales,
                        "duracion_s": round(self.duracion_total_s, 2),
                    }
                )
                self._lf.flush()
            except Exception:
                pass
        return self.guardar()
'''

OBS_DIR = PROJECT_DIR / "observability"
(OBS_DIR / "lf_tracer.py").write_text(LF_TRACER_PY.strip())
print("✓ observability/lf_tracer.py generado")

## Celda 4 — Gestión de prompts con Langfuse

Migra el prompt de síntesis de `synthesizer.py` a Langfuse Prompt Management.
El código tiene un fallback al prompt hardcodeado si Langfuse no está disponible.

In [ ]:
# Paso 1: crea el prompt en Langfuse (solo si estás conectado)
PROMPT_SINTESIS_V1 = """Eres un analista senior de vigilancia tecnológica.
A partir de los hallazgos de investigación siguientes, genera un informe
profesional y estructurado sobre el tema.

TEMA: {{tema}}

HALLAZGOS:
{{hallazgos}}

INSTRUCCIONES:
- Escribe en español, tono analítico y directo
- Estructura: Resumen ejecutivo → Hallazgos por área → Conclusiones → Fuentes
- Sintetiza e interpreta, no copies literalmente
- Señala tendencias clave, actores relevantes y gaps de información
- Formato Markdown limpio y legible"""

if LANGFUSE_ENABLED:
    try:
        # Crea el prompt en Langfuse si no existe
        prompt_obj = langfuse.create_prompt(
            name="sintesis_informe",
            prompt=PROMPT_SINTESIS_V1,
            labels=["production"],   # marca como versión activa
            config={"modelo": "gpt-4o-mini", "temperatura": 0.4},
        )
        print(f"✓ Prompt 'sintesis_informe' creado en Langfuse (versión {prompt_obj.version})")
        print(f"  Edítalo en: {LANGFUSE_HOST}/prompts")
    except Exception as e:
        print(f"⚠  No se pudo crear el prompt: {e}")
        print("   (puede que ya exista — revisa el panel)")
else:
    print("[Modo local] Prompt que se subiría a Langfuse:")
    print(f"  Nombre: sintesis_informe")
    print(f"  Variables: {{{{tema}}}}, {{{{hallazgos}}}}")
    print(f"  Labels: [production]")

In [ ]:
# Paso 2: synthesizer.py actualizado que recupera el prompt de Langfuse
SYNTHESIZER_V2_PY = '''
"""Sintetizador con prompt gestionado por Langfuse (con fallback local)."""
import os
from openai import OpenAI

MODELO_SINTESIS = "gpt-4o-mini"

# Prompt de fallback (se usa si Langfuse no está disponible)
_PROMPT_FALLBACK = """Eres un analista senior de vigilancia tecnológica.
A partir de los hallazgos siguientes, genera un informe profesional sobre:

TEMA: {tema}

HALLAZGOS:
{hallazgos}

Escribe en español. Estructura: Resumen ejecutivo → Hallazgos → Conclusiones → Fuentes.
Formato Markdown."""


def _obtener_prompt(tema: str, hallazgos: str) -> str:
    """Recupera el prompt de Langfuse o usa el fallback local."""
    pk = os.getenv("LANGFUSE_PUBLIC_KEY")
    sk = os.getenv("LANGFUSE_SECRET_KEY")
    if pk and sk:
        try:
            from langfuse import Langfuse
            lf = Langfuse(
                public_key=pk, secret_key=sk,
                host=os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com"),
            )
            # cache_ttl_seconds: no llama a Langfuse en cada request
            p = lf.get_prompt("sintesis_informe", cache_ttl_seconds=300)
            return p.compile(tema=tema, hallazgos=hallazgos)
        except Exception:
            pass
    # Fallback: prompt hardcodeado con .format()
    return _PROMPT_FALLBACK.format(tema=tema, hallazgos=hallazgos)


def generar_informe(tema: str, resultados: list[dict]) -> tuple[str, int, int]:
    """
    Genera el informe final.
    Devuelve (informe, tokens_input, tokens_output) para el tracer.
    """
    partes = []
    for r in resultados:
        fuentes_str = "\\n".join(f"  - {f}" for f in r.get("fuentes", []))
        partes.append(
            f"### Subtarea {r[\'subtarea_id\']}: {r[\'descripcion\']}\\n"
            f"{r[\'hallazgo\']}\\n"
            f"Fuentes:\\n{fuentes_str or \'  (sin fuentes externas)\'}"
        )
    hallazgos_txt = "\\n\\n".join(partes)

    prompt_txt = _obtener_prompt(tema, hallazgos_txt)

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp   = client.chat.completions.create(
        model=MODELO_SINTESIS,
        messages=[{"role": "user", "content": prompt_txt}],
        temperature=0.4,
        max_tokens=2000,
    )
    informe = resp.choices[0].message.content
    t_in    = resp.usage.prompt_tokens
    t_out   = resp.usage.completion_tokens
    return informe, t_in, t_out
'''

(PROJECT_DIR / "agent" / "synthesizer.py").write_text(SYNTHESIZER_V2_PY.strip())
print("✓ agent/synthesizer.py actualizado (recupera prompt de Langfuse con fallback)")
print()
print("Cambio clave:")
print("  generar_informe() ahora devuelve (informe, tokens_input, tokens_output)")
print("  → tokens reales de OpenAI en lugar de estimaciones")

## Celda 5 — `agent.py` integrado con `LangfuseTracer`

Actualiza el agente para usar `LangfuseTracer` y tokens reales del sintetizador.

In [ ]:
AGENT_LF_PY = '''
"""ResearchAgent con Langfuse integrado."""
import time
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import Callable

from agent.planner     import crear_plan
from agent.executor    import ejecutar_subtarea
from agent.synthesizer import generar_informe
from observability.lf_tracer import LangfuseTracer
from observability.logger    import configurar_logger


@dataclass
class ResultadoInvestigacion:
    tema:         str
    plan:         list[dict]
    resultados:   list[dict]
    informe:      str
    duracion_s:   float
    n_fuentes:    int
    ruta_informe: str  = ""
    trace_id:     str  = ""
    coste_usd:    float = 0.0


class ResearchAgent:

    def __init__(
        self,
        informes_dir: str = "./informes",
        trazas_dir:   str = "./trazas",
        log_file:     str | None = None,
        verbose:      bool = True,
    ):
        self.informes_dir = Path(informes_dir)
        self.trazas_dir   = Path(trazas_dir)
        self.informes_dir.mkdir(parents=True, exist_ok=True)
        self.trazas_dir.mkdir(parents=True, exist_ok=True)
        self.verbose = verbose
        self.logger  = configurar_logger("research_agent", log_file=log_file)

    def _log(self, msg: str):
        if self.verbose:
            print(msg)

    def investigar(
        self,
        tema: str,
        on_progreso: Callable[[int, int, str], None] | None = None,
        tags: list[str] | None = None,
    ) -> ResultadoInvestigacion:

        tracer = LangfuseTracer(
            tema=tema,
            trazas_dir=str(self.trazas_dir),
            tags=tags or [],
        )
        self._log(f"\n🔍 [{tracer.trace_id}] {tema[:60]}\n")

        # ── Fase 1: Planificación ─────────────────────────────────────────────
        self._log("📋 Fase 1/3 — Planificando...")
        with tracer.span("planificar") as s:
            plan       = crear_plan(tema)
            s.tokens_input  = 500
            s.tokens_output = len(plan) * 80
        self._log(f"   {len(plan)} subtareas")

        # ── Fase 2: Ejecución ─────────────────────────────────────────────────
        self._log("\n🔎 Fase 2/3 — Investigando...")
        resultados = []
        for i, subtarea in enumerate(plan, 1):
            with tracer.span(f"subtarea_{i}") as s:
                resultado = ejecutar_subtarea(subtarea)
                # Usa tokens del resultado si executor los devuelve, sino estima
                s.tokens_input  = resultado.get("tokens_input_total",
                                                len(subtarea["descripcion"]) * 4)
                s.tokens_output = resultado.get("tokens_output_total",
                                                len(resultado["hallazgo"]) // 4)
            resultados.append(resultado)
            if on_progreso:
                on_progreso(i, len(plan), subtarea["descripcion"])
            self._log(f"   {i}/{len(plan)} — {resultado[\'n_pasos\']} pasos")

        # ── Fase 3: Síntesis ──────────────────────────────────────────────────
        self._log("\n✍️  Fase 3/3 — Sintetizando...")
        with tracer.span("sintetizar") as s:
            # generar_informe ahora devuelve tokens reales
            informe, t_in, t_out = generar_informe(tema, resultados)
            s.tokens_input  = t_in
            s.tokens_output = t_out

        # ── Guardar informe ───────────────────────────────────────────────────
        duracion_s = tracer.duracion_total_s
        n_fuentes  = sum(len(r["fuentes"]) for r in resultados)
        fecha      = datetime.now().strftime("%Y%m%d_%H%M")
        slug       = tema[:40].replace(" ", "_").replace("/", "-")
        ruta       = self.informes_dir / f"{fecha}_{slug}.md"

        cabecera = (
            f"---\ntema: {tema}\ntrace_id: {tracer.trace_id}\n"
            f"coste_usd: {tracer.coste_total_usd:.6f}\n"
            f"duracion_s: {duracion_s:.1f}\nn_fuentes: {n_fuentes}\n---\n\n"
        )
        ruta.write_text(cabecera + informe, encoding="utf-8")

        ruta_traza = tracer.cerrar(
            informe=informe,
            coste_usd=tracer.coste_total_usd,
        )

        self._log(
            f"\n✅ [{tracer.trace_id}]  {duracion_s:.1f}s  "
            f"${tracer.coste_total_usd:.5f}  {tracer.tokens_totales:,} tok"
        )

        return ResultadoInvestigacion(
            tema=tema, plan=plan, resultados=resultados,
            informe=informe, duracion_s=duracion_s, n_fuentes=n_fuentes,
            ruta_informe=str(ruta),
            trace_id=tracer.trace_id,
            coste_usd=tracer.coste_total_usd,
        )
'''

(PROJECT_DIR / "agent" / "agent.py").write_text(AGENT_LF_PY.strip())
print("✓ agent/agent.py actualizado con LangfuseTracer")

## Celda 6 — Ejecución con Langfuse activo

In [ ]:
import importlib
import agent.agent, agent.synthesizer
importlib.reload(agent.synthesizer)
importlib.reload(agent.agent)

from agent.agent import ResearchAgent
from IPython.display import Markdown, display

agente = ResearchAgent(
    informes_dir=str(PROJECT_DIR / "informes"),
    trazas_dir=str(PROJECT_DIR / "trazas"),
    verbose=True,
)

resultado = agente.investigar(
    "Inteligencia artificial en la cadena de valor del vino gallego",
    tags=["semana-18", "vino", "agrifood"],
)

print(f"\nTrace ID:  {resultado.trace_id}")
print(f"Coste:     ${resultado.coste_usd:.5f}")
print(f"Duración:  {resultado.duracion_s:.1f}s")

if LANGFUSE_ENABLED:
    print(f"\nVer traza: {LANGFUSE_HOST}/traces/{resultado.trace_id}")

In [ ]:
# Muestra el informe
display(Markdown(resultado.informe))

## Celda 7 — Añadir puntuación de calidad a la traza

Puntúa el informe con un LLM-as-judge (el mismo patrón de la semana 8)
y vincula la puntuación a la traza en Langfuse.

In [ ]:
import json
from openai import OpenAI

PROMPT_EVALUAR = """Evalúa la calidad de este informe de vigilancia tecnológica.

INFORME:
{informe}

Puntúa de 0.0 a 1.0 en cada dimensión y da una justificación breve.
Responde SOLO con JSON válido:
{{"relevancia": 0.0, "profundidad": 0.0, "estructura": 0.0,
  "media": 0.0, "comentario": "..."}}"""

def evaluar_informe(informe: str) -> dict:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user",
                   "content": PROMPT_EVALUAR.format(informe=informe[:3000])}],
        response_format={"type": "json_object"},
        temperature=0.1,
    )
    return json.loads(resp.choices[0].message.content)

print("Evaluando calidad del informe...")
puntuacion = evaluar_informe(resultado.informe)

print(f"\nPuntuaciones:")
for k, v in puntuacion.items():
    if k != "comentario":
        barra = "█" * int(v * 20) + "░" * (20 - int(v * 20))
        print(f"  {k:<15} {barra} {v:.2f}")
print(f"\nComentario: {puntuacion.get('comentario', '')}")

# Envía la puntuación a Langfuse si está conectado
if LANGFUSE_ENABLED:
    trace = langfuse.trace(id=resultado.trace_id)
    for dim in ["relevancia", "profundidad", "estructura"]:
        if dim in puntuacion:
            trace.score(
                name=dim,
                value=float(puntuacion[dim]),
                comment=puntuacion.get("comentario", ""),
            )
    trace.score(name="calidad_media", value=float(puntuacion.get("media", 0)))
    langfuse.flush()
    print(f"\n✓ Puntuaciones enviadas a Langfuse")
    print(f"  Ver: {LANGFUSE_HOST}/traces/{resultado.trace_id}")

## Reto — Comparar dos versiones del prompt de síntesis

El poder real del versionado de prompts es comparar resultados entre versiones.

**Crea una segunda versión del prompt** en Langfuse con estas diferencias:
- Añade la instrucción: *"Incluye siempre una sección 'Implicaciones para el sector' al final"*
- Cambia el tono a: *"Escribe para un directivo no técnico"*

Luego ejecuta el agente dos veces con el mismo tema: una con la versión 1 (activa) y otra
con la versión 2, y compara los informes.

```python
# Para forzar una versión específica de Langfuse:
p = langfuse.get_prompt("sintesis_informe", version=2)
prompt_txt = p.compile(tema=tema, hallazgos=hallazgos)
```

Evalúa ambos informes con `evaluar_informe()` y muestra la comparación:

| Dimensión | Versión 1 | Versión 2 |
|-----------|-----------|----------|
| relevancia | 0.82 | 0.79 |
| profundidad | 0.75 | 0.71 |
| estructura | 0.88 | 0.91 |

¿Cuál versión es mejor según el juez LLM? ¿Y según tu propia lectura?

## Resumen de la semana 18

| Componente | Archivo | Qué aporta |
|-----------|---------|------------|
| LangfuseTracer | `observability/lf_tracer.py` | Extiende Tracer local con envío a Langfuse |
| Synthesizer v2 | `agent/synthesizer.py` | Recupera prompt de Langfuse + tokens reales |
| Agent con tags | `agent/agent.py` | Tags para filtrar trazas en el panel |
| Evaluación calidad | Celda 7 | LLM-as-judge vinculado a la traza |
| Comparación prompts | Reto | A/B test de versiones de prompt |

**Bloque 7 casi completo.** La semana 19 arranca el Bloque 8: estrategia freelance —
cómo posicionarte, fijar precios, conseguir el primer cliente y construir el portfolio.